In [1]:
# This is boilerplate code to correctly setup the settings to for notebook.
import os, sys
for root, dirs, files in os.walk(os.getcwd()):
    # print(root)

    if "pion-argon-xs-analysis" in root.split("/"):
        pypath = root.split("pion-argon-xs-analysis")[0] + "pion-argon-xs-analysis/analysis"
        print(pypath)
        break

sys.path.insert(0, pypath)

from python.analysis.NotebookUtils import init_notebook
%init_notebook

import awkward as ak
import numpy as np

from iminuit import minimize
from python.analysis import cross_section, Tags, Plots, Master, vector, SelectionTools, BeamParticleSelection
from apps.cex_analysis_input import BeamPionSelection
from apps import cex_selection_studies, cex_analysis_input
cross_section.PlotStyler.SetPlotStyle(extend_colors = True, dpi = 100)

/home/suw/pion-argon-xs-analysis/analysis
/home/suw/pion-argon-xs-analysis/analysis
env: PYTHONPATH=/home/suw/pion-argon-xs-analysis/analysis


In [2]:
args_pion = cross_section.ApplicationArguments.ResolveConfig(cross_section.LoadConfiguration("/home/suw/pion-argon-xs-analysis/analysis/work/cex_analysis_2GeV_config.json"))
args_pion.ntuple_files["mc"][0]

{'file': '/data/dune/common/PDSPAnalyzer_Ntuples/PDSPProd4a_MC_2GeV_sce_datadriven_ntuple_v09_81_00d01_set0.root',
 'type': 'PDSPAnalyser',
 'pmom': 1}

In [3]:
file_names = {k : v[0]["file"] for k, v in args_pion.ntuple_files.items()}
samples_pion = {k : Master.Data(v, nTuple_type= args_pion.ntuple_files[k][0]["type"], target_momentum = args_pion.ntuple_files[k][0]["pmom"]) for k, v in file_names.items()}

In [4]:
args_pion.selection_masks["mc"]["fiducial"]

{np.str_('/data/dune/common/PDSPAnalyzer_Ntuples/PDSPProd4a_MC_2GeV_reco1_sce_datadriven_v1_ntuple_v09_41_00_03.root'): {'TrueFiducialCut': <Array [False, True, False, True, ..., False, False, True] type='141548 * bool'>,
  'APA3Cut': <Array [False, False, False, True, ..., True, False, True] type='141548 * bool'>},
 np.str_('/data/dune/common/PDSPAnalyzer_Ntuples/PDSPProd4a_MC_2GeV_sce_datadriven_ntuple_v09_81_00d01_set0.root'): {'TrueFiducialCut': <Array [True, False, True, True, ..., True, True, False] type='131266 * bool'>,
  'APA3Cut': <Array [True, True, True, True, ..., False, True, False] type='131266 * bool'>},
 np.str_('/data/dune/common/PDSPAnalyzer_Ntuples/PDSPProd4a_MC_2GeV_sce_datadriven_ntuple_v09_81_00d01_set1.root'): {'TrueFiducialCut': <Array [True, True, True, False, ..., True, True, True] type='130635 * bool'>,
  'APA3Cut': <Array [True, True, False, False, ..., False, True, True] type='130635 * bool'>},
 np.str_('/data/dune/common/PDSPAnalyzer_Ntuples/PDSPProd4a_MC

In [5]:
data_selected = cex_analysis_input.BeamPionSelection(samples_pion["data"], args_pion, False)
mc_selected = cex_analysis_input.BeamPionSelection(samples_pion["mc"], args_pion, True)

f=<Array [False, False, False, ..., False, True, False] type='1349399 * bool'>
f=<Array [False, False, False, ..., False, False, False] type='263960 * ?bool'>
f=<Array [[True, True, ..., False, False], ...] type='63521 * option[var * bool]'>


'BeamPionSelection' executed in 0.0835s

f=<Array [True, False, True, True, ..., False, True, False] type='131266 * bool'>
f=<Array [False, False, True, True, ..., False, True, False] type='52654 * ?bool'>
f=<Array [[True, True, ..., True, True], ...] type='21372 * option[var * bool]'>


'BeamPionSelection' executed in 0.0290s

In [7]:
args_pion.selection_masks["mc"]["beam"][file_names["mc"]]
args_pion.selection_masks["data"]["beam"][file_names["data"]]

{'TrueFiducialCut': <Array [True, True, True, True, ..., True, True, True] type='263960 * bool'>,
 'PiBeamSelection': <Array [False, False, True, ..., False, False, False] type='263960 * bool'>,
 'PandoraTagCut': <Array [True, True, True, True, ..., True, True, True] type='263960 * bool'>,
 'CaloSizeCut': <Array [True, True, True, True, ..., True, True, True] type='263960 * bool'>,
 'HasFinalStatePFOsCut': <Array [False, True, False, True, ..., True, True, True] type='263960 * bool'>,
 'APA3Cut': <Array [True, True, True, True, ..., True, True, True] type='263960 * bool'>,
 'DxyCut': <Array [False, True, True, True, ..., False, True, False] type='263960 * ?bool'>,
 'CosThetaCut': <Array [False, True, True, True, ..., True, True, False] type='263960 * ?bool'>,
 'MichelScoreCut': <Array [True, True, True, True, ..., True, True, True] type='263960 * bool'>,
 'MedianDEdXCut': <Array [True, True, True, True, ..., True, False, True] type='263960 * bool'>,
 'BeamScraperCut': <Array [False, Tr

In [8]:
args_pion.selection_masks["data"]["pi"][file_names["data"]]

{'Chi2ProtonSelection': <Array [[False, False, ..., True, True], ...] type='63521 * option[var * bool]'>,
 'TrackScoreCut': <Array [[False, True, ..., False, False], ...] type='63521 * option[var * b...'>,
 'NHitsCut': <Array [[True, True, ..., True, True], ...] type='63521 * option[var * bool]'>,
 'PiPlusSelection': <Array [[False, False, True, ..., False, False], ...] type='63521 * var * bool'>}

In [16]:
n_pi = SelectionTools.GetPFOCounts(args_pion.selection_masks["mc"]["pi"][file_names["mc"]])
n_pi

<Array [1, 2, 1, 0, 0, 1, 0, 0, ..., 0, 1, 1, 0, 1, 0, 0] type='21372 * ?int64'>

In [13]:
reco_regions = cex_analysis_input.RegionSelection(data_selected, args_pion, False)
reco_regions

'RegionSelection' executed in 24.6068s

{'absorption': <Array [False, False, False, ..., False, False, False] type='63521 * bool'>,
 'charge_exchange': <Array [False, False, False, ..., False, False, False] type='63521 * bool'>,
 'pion_production': <Array [False, False, False, False, ..., True, False, True] type='63521 * bool'>}

In [14]:
reco_regions, true_regions = cex_analysis_input.RegionSelection(samples_pion["mc"], args_pion, True)

f=<Array [True, False, True, True, ..., False, True, False] type='131266 * bool'>


'RegionSelection' executed in 25.4000s

In [17]:
from python.analysis import RegionDefinitions


ri={}
for r in RegionDefinitions.regions:
    ri[r] = {"reco_regions" : reco_regions, "true_regions" : true_regions}

ri

{'default': {'reco_regions': {'absorption': <Array [False, False, False, ..., False, False, False] type='21372 * bool'>,
   'charge_exchange': <Array [False, False, False, ..., False, False, True] type='21372 * bool'>,
   'pion_production': <Array [True, True, True, False, ..., True, False, False] type='21372 * bool'>},
  'true_regions': {'absorption': <Array [False, False, False, ..., False, False, False] type='21372 * bool'>,
   'charge_exchange': <Array [False, False, False, ..., False, False, True] type='21372 * bool'>,
   'pion_production': <Array [True, True, True, True, ..., True, True, False] type='21372 * bool'>}},
 'pdsp_1GeV_regions': {'reco_regions': {'absorption': <Array [False, False, False, ..., False, False, False] type='21372 * bool'>,
   'charge_exchange': <Array [False, False, False, ..., False, False, True] type='21372 * bool'>,
   'pion_production': <Array [True, True, True, False, ..., True, False, False] type='21372 * bool'>},
  'true_regions': {'absorption': <Ar